# BreizhCrops: EDA

One sample = one agricultural field parcel in Brittany, France. Each parcel has a 12-month Sentinel-2 L1C time series (resampled from the raw variable-length acquisitions to 12 monthly steps, plus a computed NDVI). The task is **crop-type classification** across 9 classes, with **geographic holdout** by NUTS-3 region (frh01–frh04 and Belle-Île).

The dataset is **S2-only** — no Sentinel-1 SAR or climate data accompany the S2 reflectance (those arrays are zero-filled by the loader).

## Setup

In [3]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# find the repo root by walking up to the folder that has data/
REPO = Path.cwd()
while not (REPO / 'data').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
print('repo:', REPO)

# register src so we can import the loader
BZ = REPO / 'data' / 'input' / 'benchmarks' / 'breizhcrops'
print('BreizhCrops root:', BZ)

# Use the breizhcrops package to explore regions
import breizhcrops as bzh
BZ_REGIONS = ['frh01', 'frh02', 'frh03', 'frh04', 'belle-ile']
print('Regions:', BZ_REGIONS)

repo: /Users/akshithchowdary/Developer/Projects/org/abe/robustness
BreizhCrops root: /Users/akshithchowdary/Developer/Projects/org/abe/robustness/data/input/benchmarks/breizhcrops


ModuleNotFoundError: No module named 'breizhcrops'

## Raw layout of one sample

The breizhcrops package returns each parcel as a `(T, 13)` array in this **alphabetical** band order (L1C). The loader keeps 10 S2 spectral bands + a computed NDVI, dropping B1 and B10, and resamples to exactly 12 timesteps.

| idx | Band | Source | Used by code? |
|:---:|------|--------|:-------------:|
| 0 | B1 | Sentinel-2 MSI (Coastal aerosol, 60m) | ❌ Skipped |
| 1 | B10 | Sentinel-2 MSI (Cirrus, 60m) | ❌ Skipped |
| 2 | B11 | Sentinel-2 MSI (SWIR 1, 20m) | ✅ S2 |
| 3 | B12 | Sentinel-2 MSI (SWIR 2, 20m) | ✅ S2 |
| 4 | B2 | Sentinel-2 MSI (Blue, 10m) | ✅ S2 |
| 5 | B3 | Sentinel-2 MSI (Green, 10m) | ✅ S2 |
| 6 | B4 | Sentinel-2 MSI (Red, 10m) | ✅ S2 |
| 7 | B5 | Sentinel-2 MSI (Red Edge 1, 20m) | ✅ S2 |
| 8 | B6 | Sentinel-2 MSI (Red Edge 2, 20m) | ✅ S2 |
| 9 | B7 | Sentinel-2 MSI (Red Edge 3, 20m) | ✅ S2 |
| 10 | B8 | Sentinel-2 MSI (NIR, 10m) | ✅ S2 |
| 11 | B8A | Sentinel-2 MSI (Narrow NIR, 20m) | ✅ S2 |
| 12 | B9 | Sentinel-2 MSI (Water vapour, 60m) | ❌ Skipped |

The code's `BZ_S2_KEEP` selects indices `[4,5,6,7,8,9,10,11,2,3]` corresponding to B2..B8A, B11, B12 then appends NDVI, yielding **11 S2 bands**. S1 and climate modalities are zero-filled — BreizhCrops is **S2-only**.

In [ ]:
# Load one region and inspect a few samples
ds = bzh.BreizhCrops('frh01', root=str(BZ), level='L1C', load_timeseries=True, verbose=False)
print(f'frh01 dataset size: {len(ds)} parcels')

x, y, _ = ds[0]
x = np.asarray(x, dtype=np.float32)
print(f'Parcel 0: shape {x.shape}, label={y}')

# Show class names
import json
cm_path = BZ / 'frh01' / 'classmapping.csv'
if cm_path.exists():
    import csv
    with open(cm_path) as f:
        reader = csv.reader(f)
        classes = {int(r[1]): r[2] for r in reader if len(r) >= 3 and r[1].strip().isdigit()}
    print('\nCrop classes (id -> name):')
    for cid, cname in sorted(classes.items()):
        print(f'  {cid}: {cname}')
else:
    print('\n(no classmapping.csv found for raw inspection)')
    
# Show raw reflectance values for one parcel: timesteps as rows, bands as columns
BZ_X_BANDS = ['B1','B10','B11','B12','B2','B3','B4','B5','B6','B7','B8','B8A','B9']
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
t = pd.DataFrame(x[:12], index=months, columns=BZ_X_BANDS)
t.round(2)

## What this one parcel looks like over the season

Reflectance at raw sensor DN scale (the loader multiplies by 1e4). We show the kept S2 bands + NDVI.

In [ ]:
b4, b8 = x[:, 6], x[:, 10]  # B4, B8 in the alphabetical ordering
denom = b8 + b4
ndvi = np.divide(b8 - b4, denom, out=np.zeros_like(b4), where=denom != 0)

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
s2_cols = [4,5,6,7,8,9,10,11,2,3]  # B2..B8A, B11, B12 in alphabetical indexing
s2_names = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
for c, n in zip(s2_cols, s2_names):
    ax[0].plot(months, x[:12, c], marker='o', label=n)
ax[0].set_title('Sentinel-2 reflectance (DN)'); ax[0].set_ylabel('reflectance'); ax[0].legend(ncol=5, fontsize=8)

ax[1].plot(months, ndvi[:12], marker='o', color='green')
ax[1].set_title('NDVI'); ax[1].set_ylabel('NDVI'); ax[1].axhline(0, color='grey', lw=0.6); ax[1].set_xlabel('month')
fig.suptitle(f'one frh01 parcel  (class {y})')
plt.tight_layout(); plt.show()

## Dataset-level: class balance and region composition

Use the loader (not the raw breizhcrops package) to see the final benchmark structure with all 5 regions.

In [ ]:
from evals.benchmarks.breizhcrops import load_benchmark as load_breizhcrops
bench = load_breizhcrops(root=REPO / 'data' / 'input' / 'benchmarks', shuffle=False)
print(f'Total parcels: {bench.n_samples}')
print(f'Timesteps (per parcel): {len(bench.native.s2.values[0])}')
print(f'S2 bands: {bench.s2_bands}')
print(f'Label names: {bench.label_names}')
print(f'Years present: {sorted(set(bench.years.tolist()))}')

# Region (group) counts
import collections
region_counts = collections.Counter(bench.groups.tolist())
print('\nParcels per region (holdout group):')
for reg, cnt in sorted(region_counts.items()):
    print(f'  {reg}: {cnt}')

# Class balance
class_counts = collections.Counter(bench.labels.tolist())
print('\nClass balance:')
for cid, cnt in sorted(class_counts.items()):
    name = bench.label_names[cid] if bench.label_names and cid < len(bench.label_names) else '?'
    print(f'  {cid} ({name}): {cnt}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
regs, cnts = zip(*sorted(region_counts.items()))
ax[0].bar(regs, cnts, color=['#2563eb','#16a34a','#f97316','#a855f7','#ef4444'])
ax[0].set_title('parcels per region'); ax[0].tick_params(rotation=45)

classes_sorted = sorted(class_counts.items())
ax[1].bar([f'{c}\n({bench.label_names[c][:8] if bench.label_names and c < len(bench.label_names) else "?"})' for c, _ in classes_sorted],
          [v for _, v in classes_sorted], color='#0ea5e9')
ax[1].set_title('class distribution'); ax[1].tick_params(rotation=60)
plt.tight_layout(); plt.show()

## Region-wise class composition

Each region has a different class distribution — trains on regions frh01–frh03, tests on frh04 and Belle-Île (the paper's protocol).

In [ ]:
region_class = collections.Counter(zip(bench.groups.tolist(), bench.labels.tolist()))
pivot = pd.DataFrame(
    {(r, c): region_class.get((r, c), 0) for r in sorted(region_counts) for c in sorted(class_counts)}
).fillna(0).astype(int)
pivot.index = ['class ' + str(c) for c in sorted(class_counts)]
pivot.T.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='tab10')
plt.title('class composition by region'); plt.ylabel('parcels'); plt.legend(loc='upper right', fontsize=7)
plt.tight_layout(); plt.show()

## Takeaways

One sample is a 12-step S2-only parcel time series with 11 bands (10 spectral + NDVI). Label = crop type (9 classes). `groups` = NUTS-3 region, making this a strict geographic holdout benchmark (train on frh01–frh03, test on frh04 and Belle-Île). All data is from 2017. No latlon, S1, or climate data available.